# DDRI Representative-Station Prediction Dataset Builder

대표 대여소 15개만 대상으로 예측용 long-format 데이터셋을 생성한다.

- 단위 행: `station_id + date + hour`
- 타깃: `rental_count`
- 피처: `cluster`, `mapped_dong_code`, `weekday`, `month`, `holiday`, `temperature`, `humidity`, `precipitation`, `wind_speed`


In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
SHARED_DIR = ROOT / '3조 공유폴더'
TRAIN_CLUSTER_PATH = ROOT / 'works/01_clustering/08_integrated/final/results/second_clustering_results/data/ddri_second_cluster_train_with_labels.csv'
TEST_CLUSTER_PATH = ROOT / 'works/01_clustering/08_integrated/final/results/second_clustering_results/data/ddri_second_cluster_test_with_labels.csv'
OUTPUT_DIR = ROOT / 'works/05_prediction_long/data'

TRAIN_RENTAL_DIR = SHARED_DIR / '2023 강남구 따릉이 이용정보'
TRAIN_RENTAL_DIR_2024 = SHARED_DIR / '2024 강남구 따릉이 이용정보'
TEST_RENTAL_DIR = SHARED_DIR / '2025 강남구 따릉이 이용정보'

WEATHER_PATHS = {
    2023: SHARED_DIR / '2023-2024년 강남구 날씨데이터(00시-24시)/gangnam_weather_1year_2023.csv',
    2024: SHARED_DIR / '2023-2024년 강남구 날씨데이터(00시-24시)/gangnam_weather_1year_2024.csv',
    2025: SHARED_DIR / '2024년 강남구 날씨데이터(00시-24시)/gangnam_weather_1year_2025.csv',
}

REPRESENTATIVE_STATIONS = {
    4908: '업무/상업 혼합형',
    2328: '업무/상업 혼합형',
    4902: '업무/상업 혼합형',
    2377: '아침 도착 업무 집중형',
    2323: '아침 도착 업무 집중형',
    2348: '아침 도착 업무 집중형',
    2312: '주거 도착형',
    2354: '주거 도착형',
    4917: '주거 도착형',
    2321: '생활권 혼합형',
    2320: '생활권 혼합형',
    3616: '생활권 혼합형',
    3643: '외곽 주거형',
    2359: '외곽 주거형',
    2392: '외곽 주거형',
}

BASE_COLUMNS = ['station_id', 'station_name', 'mapped_dong_code', 'cluster']

In [ ]:
HOLIDAYS = {
    pd.Timestamp('2023-01-01'), pd.Timestamp('2023-01-21'), pd.Timestamp('2023-01-22'), pd.Timestamp('2023-01-23'), pd.Timestamp('2023-01-24'),
    pd.Timestamp('2023-03-01'), pd.Timestamp('2023-05-05'), pd.Timestamp('2023-05-27'), pd.Timestamp('2023-06-06'), pd.Timestamp('2023-08-15'),
    pd.Timestamp('2023-09-28'), pd.Timestamp('2023-09-29'), pd.Timestamp('2023-09-30'), pd.Timestamp('2023-10-03'), pd.Timestamp('2023-10-09'),
    pd.Timestamp('2023-12-25'),
    pd.Timestamp('2024-01-01'), pd.Timestamp('2024-02-09'), pd.Timestamp('2024-02-10'), pd.Timestamp('2024-02-11'), pd.Timestamp('2024-02-12'),
    pd.Timestamp('2024-03-01'), pd.Timestamp('2024-04-10'), pd.Timestamp('2024-05-05'), pd.Timestamp('2024-05-06'), pd.Timestamp('2024-05-15'),
    pd.Timestamp('2024-06-06'), pd.Timestamp('2024-08-15'), pd.Timestamp('2024-09-16'), pd.Timestamp('2024-09-17'), pd.Timestamp('2024-09-18'),
    pd.Timestamp('2024-10-01'), pd.Timestamp('2024-10-03'), pd.Timestamp('2024-10-09'), pd.Timestamp('2024-12-25'),
    pd.Timestamp('2025-01-01'), pd.Timestamp('2025-01-28'), pd.Timestamp('2025-01-29'), pd.Timestamp('2025-01-30'), pd.Timestamp('2025-03-01'),
    pd.Timestamp('2025-03-03'), pd.Timestamp('2025-05-05'), pd.Timestamp('2025-05-06'), pd.Timestamp('2025-06-06'), pd.Timestamp('2025-08-15'),
    pd.Timestamp('2025-10-03'), pd.Timestamp('2025-10-05'), pd.Timestamp('2025-10-06'), pd.Timestamp('2025-10-07'), pd.Timestamp('2025-10-08'),
    pd.Timestamp('2025-10-09'), pd.Timestamp('2025-12-25'),
}

In [ ]:
def collect_csv_files(directory: Path) -> list[Path]:
    return sorted(directory.glob('*.csv'))


def load_base(cluster_path: Path) -> pd.DataFrame:
    base = pd.read_csv(cluster_path)[BASE_COLUMNS].copy()
    base = base[base['station_id'].isin(REPRESENTATIVE_STATIONS)].copy()
    base['station_group'] = base['station_id'].map(REPRESENTATIVE_STATIONS)
    return base.sort_values('station_id').reset_index(drop=True)


def load_weather(years: list[int]) -> pd.DataFrame:
    frames = []
    for year in years:
        weather = pd.read_csv(WEATHER_PATHS[year]).copy()
        weather['datetime'] = pd.to_datetime(weather['datetime'])
        frames.append(weather)
    return pd.concat(frames, ignore_index=True)


def aggregate_rentals(files: list[Path], station_ids: set[int]) -> pd.DataFrame:
    grouped_frames = []
    for file_path in files:
        for chunk in pd.read_csv(file_path, usecols=['대여일시', '대여 대여소번호'], chunksize=200_000):
            rental_dt = pd.to_datetime(chunk['대여일시'], errors='coerce').dt.floor('h')
            station_id = pd.to_numeric(chunk['대여 대여소번호'], errors='coerce')
            valid = pd.DataFrame({'station_id': station_id.astype('Int64'), 'datetime': rental_dt}).dropna()
            if valid.empty:
                continue
            valid['station_id'] = valid['station_id'].astype('int64')
            valid = valid[valid['station_id'].isin(station_ids)]
            if valid.empty:
                continue
            grouped = valid.groupby(['station_id', 'datetime']).size().reset_index(name='rental_count')
            grouped_frames.append(grouped)
    if not grouped_frames:
        return pd.DataFrame(columns=['station_id', 'datetime', 'rental_count'])
    counts = pd.concat(grouped_frames, ignore_index=True)
    return counts.groupby(['station_id', 'datetime'], as_index=False)['rental_count'].sum()


def build_long_dataset(base_df: pd.DataFrame, files: list[Path], years: list[int], output_path: Path) -> pd.DataFrame:
    station_ids = set(base_df['station_id'])
    start = pd.Timestamp(f'{years[0]}-01-01 00:00:00')
    end = pd.Timestamp(f'{years[-1]}-12-31 23:00:00')
    full_hours = pd.date_range(start, end, freq='h')
    rental_counts = aggregate_rentals(files, station_ids)
    full_index = pd.MultiIndex.from_product([sorted(station_ids), full_hours], names=['station_id', 'datetime'])
    rental_counts = rental_counts.set_index(['station_id', 'datetime']).reindex(full_index, fill_value=0).reset_index()
    rental_counts['date'] = rental_counts['datetime'].dt.date.astype(str)
    rental_counts['hour'] = rental_counts['datetime'].dt.hour
    rental_counts['weekday'] = rental_counts['datetime'].dt.weekday
    rental_counts['month'] = rental_counts['datetime'].dt.month
    rental_counts['holiday'] = rental_counts['datetime'].dt.normalize().isin(HOLIDAYS).astype('int8')
    weather = load_weather(years)
    result = rental_counts.merge(weather, on='datetime', how='left')
    result = result.merge(base_df, on='station_id', how='left')
    result = result[
        [
            'station_id', 'station_name', 'station_group', 'date', 'hour', 'rental_count', 'cluster', 'mapped_dong_code',
            'weekday', 'month', 'holiday', 'temperature', 'humidity', 'precipitation', 'wind_speed'
        ]
    ].sort_values(['station_id', 'date', 'hour']).reset_index(drop=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, encoding='utf-8-sig')
    return result

In [ ]:
train_base = load_base(TRAIN_CLUSTER_PATH)
test_base = load_base(TEST_CLUSTER_PATH)

train_files = collect_csv_files(TRAIN_RENTAL_DIR) + collect_csv_files(TRAIN_RENTAL_DIR_2024)
test_files = collect_csv_files(TEST_RENTAL_DIR)

train_df = build_long_dataset(
    base_df=train_base,
    files=train_files,
    years=[2023, 2024],
    output_path=OUTPUT_DIR / 'ddri_prediction_long_train_2023_2024.csv',
)

test_df = build_long_dataset(
    base_df=test_base,
    files=test_files,
    years=[2025],
    output_path=OUTPUT_DIR / 'ddri_prediction_long_test_2025.csv',
)

train_df.shape, test_df.shape

In [ ]:
train_df.head(3)

In [ ]:
test_df.head(3)

In [ ]:
train_df[['temperature', 'humidity', 'precipitation', 'wind_speed']].isna().sum(), test_df[['temperature', 'humidity', 'precipitation', 'wind_speed']].isna().sum()